In [ ]:
import pandas as pd

# === Step 1: Load files ===
refs_file = "/home/dan_pham/Public/NIPTorrent/STATISTIC/DATA/LIST_OF_REF/REFERENCES.csv"
metadata_file = "/home/dan_pham/Public/NIPTorrent/STATISTIC/DATA/METADATA1.csv"
output_file = "/home/dan_pham/Public/NIPTorrent/STATISTIC/DATA/LIST_OF_REF/REF_annotation.csv"

refs = pd.read_csv(refs_file)
metadata = pd.read_csv(metadata_file)

# === Step 2: Clean column names ===
refs.columns = refs.columns.str.strip()
metadata.columns = metadata.columns.str.strip()

# === Step 3: Extract SAMPLE ID and BARCODE from metadata ===
# Keep relevant metadata columns
meta_cols = [
    "SAMPLE ID",
    "IONXPRESS BARCODE",
    "MATERNAL AGE",
    "GA",
    "FF (%)",
    "UNIQUE READS (M)",
    "Z21",
    "Z18",
    "Z13",
    "GENDER",
    "AVERAGE READ LENGTH (bp)",
    "GC CONTENT (%)",
    "DUPLICATION (%)",
    "Nuchal Translucency (mm)"
]
metadata_subset = metadata[meta_cols].copy()

# === Step 4: Function to extract SAMPLE ID and BARCODE from REF ===
def extract_sample_barcode(ref):
    parts = str(ref).split("_")
    sample_id = parts[0]
    barcode = "IonXpress_" + parts[1] if len(parts) > 1 else ""
    return sample_id, barcode

# === Step 5: Create a new DataFrame to store aligned data ===
aligned_rows = []

for idx, row in refs.iterrows():
    new_row = {}
    for col in refs.columns:  # REF1, REF2, REF3...
        ref_val = row[col]
        sample_id, barcode = extract_sample_barcode(ref_val)
        meta_row = metadata_subset[
            (metadata_subset["SAMPLE ID"] == sample_id) &
            (metadata_subset["IONXPRESS BARCODE"] == barcode)
        ]
        # Add original REF
        new_row[col] = ref_val
        if not meta_row.empty:
            # Add metadata columns with REF prefix
            for meta_col in meta_cols[2:]:  # skip SAMPLE ID + BARCODE
                new_row[f"{col}_{meta_col}"] = meta_row.iloc[0][meta_col]
        else:
            # If no match, fill with NaN
            for meta_col in meta_cols[2:]:
                new_row[f"{col}_{meta_col}"] = pd.NA
    aligned_rows.append(new_row)

# === Step 6: Convert to DataFrame and save ===
aligned_df = pd.DataFrame(aligned_rows)
aligned_df.to_csv(output_file, index=False)

print(f"✅ Reference list aligned with metadata saved to: {output_file}")


✅ Reference list aligned with metadata saved to: /home/dan_pham/Public/NIPTorrent/STATISTIC/DATA/LIST_OF_REF/ref534_annotation.csv
